# 03 — Phase 3: Resize + Pad for DINOv2

Takes only the **accepted** crops from `review_meta.json` (Phase 2 output),  
resizes with aspect ratio preserved, detects the crop's background color,  
and pads to a 224×224 square using that color.  
Then copies the results to `final/` with the patent's first CPC code in each filename.

> **Prerequisites:** Phase 1 (extraction) and Phase 2 (review) must be complete.

In [ ]:
import sys; sys.path.insert(0, "..")
import json
from pathlib import Path
from src.config_loader import load_config
from src.processor import process_crops, organize_processed

cfg = load_config()

meta_path = Path(cfg["paths"]["logs"]) / "review_meta.json"
if not meta_path.exists():
    raise FileNotFoundError(
        f"review_meta.json not found at {meta_path}.\n"
        "Run Phase 2 (02_review.ipynb) first."
    )

with open(meta_path) as f:
    meta = json.load(f)

crops_dir      = Path(cfg["paths"]["crops"])
accepted_paths = []

for patent_id, pdata in meta.items():
    if pdata.get("is_duplicate") or pdata.get("review_status") == "DISAPPROVED":
        continue
    for fname, idata in pdata.get("images", {}).items():
        if idata.get("approved") and not idata.get("needs_split"):
            p = crops_dir / patent_id / fname
            if p.exists():
                accepted_paths.append(p)

print(f"Accepted crops to process : {len(accepted_paths)}")
print(f"Target size               : {cfg['processor']['target_size']}×{cfg['processor']['target_size']} px")
print(f"Processed output          : {cfg['paths']['processed']}")
print(f"Final output (with CPC)   : {cfg['paths']['final']}")

In [ ]:
# Resize + pad with auto-detected background color — skips already-processed files
processed_paths = process_crops(accepted_paths, cfg)

In [ ]:
# Quick visual check — show first 6 processed images
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, p in zip(axes.flat, processed_paths[:6]):
    img = Image.open(p)
    ax.imshow(img)
    ax.set_title(Path(p).name[:35], fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Copy to final/ with CPC code injected into each filename
# Output: final/{patent_id}/{patent_id}_{CPC}_p{page}_c{crop}.png
# CPC codes are read from the PatSeer Excel (paths.excel)
organize_processed(cfg)